# 01a - Regresi Logistik
Materi Modul 1: apa itu klasifikasi, pengantar regresi logistik, klasifikasi dengan regresi logistik, regresi logistik multi-kelas, dan menerapkan model regresi logistik.

Regresi logistik memodelkan **probabilitas** kelas dengan fungsi sigmoid:

$P(y=1|x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots)}}$

Dataset: **Breast Cancer** (`datasets/breast_cancer.csv`, 569 sampel, 30 fitur). Kolom `target`: **1 = ganas**, 0 = jinak.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Fungsi sigmoid

In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.plot(z, sigmoid)
plt.axhline(0.5, color='red', linestyle='--', label='Threshold 0.5')
plt.xlabel('z (kombinasi linier fitur)'); plt.ylabel('Probabilitas')
plt.title('Fungsi sigmoid'); plt.legend(); plt.show()

## 2. Ambil data & split
Pakai `stratify=y` supaya proporsi kelas di train dan test sama.

In [ ]:
df = pd.read_csv('../datasets/breast_cancer.csv')
print(df.shape, '| missing:', df.isna().sum().sum())
print(df['target'].value_counts())

X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 3. Latih model

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print('Akurasi train:', round(model.score(X_train, y_train), 4))
print('Akurasi test :', round(model.score(X_test, y_test), 4))

## 4. Prediksi kelas vs probabilitas

In [ ]:
proba = model.predict_proba(X_test)[:5]
kelas = model.predict(X_test)[:5]

hasil = pd.DataFrame(proba, columns=['P(jinak)', 'P(ganas)']).round(3)
hasil['prediksi'] = kelas
hasil['asli'] = y_test.values[:5]
hasil

## 5. Koefisien: fitur mana yang paling berpengaruh?
Koefisien positif -> menaikkan peluang ganas. Fitur sudah di-scale sehingga koefisien bisa dibandingkan.

In [ ]:
koef = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()
print(koef.head(5).round(2))
print(koef.tail(5).round(2))

pd.concat([koef.head(8), koef.tail(8)]).plot(kind='barh', figsize=(7, 5))
plt.title('Koefisien terbesar (negatif dan positif)'); plt.show()

## 6. Regularisasi pada regresi logistik (parameter `C`)
`C` = kebalikan kekuatan regularisasi. **C kecil = regularisasi kuat**, C besar = regularisasi lemah.

In [ ]:
print('C      | akurasi train | akurasi test | jumlah koef != 0 (L1)')
for c in [0.001, 0.01, 0.1, 1, 10, 100]:
    m_l2 = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(X_train, y_train)
    m_l1 = make_pipeline(StandardScaler(), LogisticRegression(C=c, penalty='l1', solver='liblinear')).fit(X_train, y_train)
    nol = np.sum(m_l1[-1].coef_ != 0)
    print(f'{c:<6} | {m_l2.score(X_train, y_train):.4f}        | {m_l2.score(X_test, y_test):.4f}       | {nol}')

## 7. Regresi logistik multi-kelas (Wine: 3 kelas)

In [ ]:
wine = pd.read_csv('../datasets/wine.csv')
Xw, yw = wine.drop(columns='target'), wine['target']
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.2, stratify=yw, random_state=42)

multi = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
multi.fit(Xw_train, yw_train)

print('Kelas       :', multi.classes_)
print('Bentuk coef :', multi[-1].coef_.shape, '(3 kelas x 13 fitur)')
print('Akurasi test:', round(multi.score(Xw_test, yw_test), 4))
print('Probabilitas sampel pertama:', multi.predict_proba(Xw_test)[0].round(3), '(jumlah = 1)')

## Catatan
- (kenapa disebut regresi tapi dipakai untuk klasifikasi, arti koefisien, pengaruh C, beda one-vs-rest dan multinomial, dll.)